In [ ]:
import pandas as pd
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"

test_ids = test["id"]

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

y = train[TARGET]
X = train.drop(columns=[TARGET])

# find categorical columns
cat_features = X.select_dtypes("object").columns.tolist()

model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    eval_metric="AUC",
    loss_function="Logloss",
    task_type="GPU",
    devices="0",
    random_seed=42,
    early_stopping_rounds=200
)

model.fit(
    X,
    y,
    cat_features=cat_features,
    verbose=200
)

pred_probs = model.predict_proba(test)[:,1]

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_catboost_gpu.csv", index=False)

/tmp/ipykernel_166175/2584916530.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes("object").columns.tolist()
Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 174ms	remaining: 14m 31s
200:	total: 29.5s	remaining: 11m 45s
400:	total: 58.6s	remaining: 11m 12s
600:	total: 1m 28s	remaining: 10m 48s
800:	total: 1m 58s	remaining: 10m 22s
1000:	total: 2m 28s	remaining: 9m 54s
1200:	total: 2m 59s	remaining: 9m 26s
1400:	total: 3m 28s	remaining: 8m 56s
1600:	total: 3m 58s	remaining: 8m 26s
1800:	total: 4m 28s	remaining: 7m 57s
2000:	total: 4m 58s	remaining: 7m 27s
2200:	total: 5m 28s	remaining: 6m 57s
2400:	total: 5m 58s	remaining: 6m 27s
2600:	total: 6m 28s	remaining: 5m 58s
2800:	total: 6m 58s	remaining: 5m 28s
3000:	total: 7m 28s	remaining: 4m 58s
3200:	total: 7m 58s	remaining: 4m 28s
3400:	total: 8m 28s	remaining: 3m 59s
3600:	total: 8m 58s	remaining: 3m 29s
3800:	total: 9m 28s	remaining: 2m 59s
4000:	total: 9m 58s	remaining: 2m 29s
4200:	total: 10m 28s	remaining: 1m 59s
4400:	total: 10m 58s	remaining: 1m 29s
4600:	total: 11m 29s	remaining: 59.8s
4800:	total: 11m 59s	remaining: 29.8s
4999:	total: 12m 29s	remaining: 0us


## Catboost Optimization

In [3]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"

test_ids = test["id"]

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])


y = train[TARGET]
X = train.drop(columns=[TARGET])

cat_features = X.select_dtypes("object").columns.tolist()


X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=8,

    eval_metric="AUC",
    loss_function="Logloss",

    task_type="GPU",
    devices="0",

    random_seed=42,

    early_stopping_rounds=200,

    verbose=200
)


model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_features
)


pred_probs = model.predict_proba(test)[:, 1]


submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_catboost_gpuV1.1.csv", index=False)

print("submission_catboost_gpuV1.1.csv created")

/tmp/ipykernel_201766/3550931150.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes("object").columns.tolist()
Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8969475	best: 0.8969475 (0)	total: 127ms	remaining: 10m 35s
200:	test: 0.9140522	best: 0.9140522 (200)	total: 25.1s	remaining: 9m 58s
400:	test: 0.9150567	best: 0.9150567 (400)	total: 49.7s	remaining: 9m 29s
600:	test: 0.9155694	best: 0.9155699 (599)	total: 1m 14s	remaining: 9m 7s
800:	test: 0.9157893	best: 0.9157893 (800)	total: 1m 40s	remaining: 8m 44s
1000:	test: 0.9159017	best: 0.9159017 (1000)	total: 2m 5s	remaining: 8m 21s
1200:	test: 0.9159759	best: 0.9159759 (1200)	total: 2m 30s	remaining: 7m 56s
1400:	test: 0.9160285	best: 0.9160285 (1400)	total: 2m 55s	remaining: 7m 32s
1600:	test: 0.9160227	best: 0.9160461 (1497)	total: 3m 21s	remaining: 7m 7s
bestTest = 0.9160461128
bestIteration = 1497
Shrink model to first 1498 iterations.
submission_catboost_gpuV1.1.csv created
